# T1: EPA Full Modeling Table

Tertiary merge per `MERGE.md` (§4, **T1**) — the terminal assembly. Attaches the
station × year context (S2) to every water-quality measurement event (S1),
yielding the single final modeling table: every WQ measurement, its full daily
climate/streamflow record, its static station geography and soil, and every
annual watershed/county/state contextual variable, in one CSV.

**Inputs** (both `data/03b_merge_secondary/...`):
- `wq-geo-soil-daily.csv` (**S1**) — one row per WQ measurement event
  (`MonitoringLocationIdentifier` + `ActivityStartDateTime`), carrying daily
  climate/streamflow + static geography/soil. This is the base grain.
- `station-year-context.csv` (**S2**) — one row per station + year, carrying land
  use, BMP, NPDES proximity, county agriculture, and state chemical spending.

**Join:** left join on `MonitoringLocationIdentifier` + `year`, where
`year = YEAR(ActivityStartDate)` derived from the measurement timestamp. S2 is
one row per station-year, so it broadcasts the annual context onto each
measurement in that station-year without changing the grain.

**Overlapping columns.** S1 and S2 both carry the station's `county_fips`,
`huc12_code`, and `huc8_code` (S1 got them from P2's geography, S2 uses them as
membership join keys). Those are dropped from the S2 side — S1's copies are kept
— so no `_x`/`_y` suffixes appear.

**Output:** `data/03c_merge_tertiary/epa-full.csv`, one row per WQ measurement
event — superseding today's `data/tabular/merged/epa-climate-merged.csv`.

In [1]:
import os

import pandas as pd

SECONDARY = "../../data/03b_merge_secondary"
OUT_DIR = "../../data/03c_merge_tertiary"
OUT_FILE = f"{OUT_DIR}/epa-full.csv"

KEY = "MonitoringLocationIdentifier"
GRAIN = [KEY, "ActivityStartDateTime"]

# Codes stay strings so leading zeros survive the round-trip.
S1_CODES = {"HUCEightDigitCode": str, "county_fips": str, "huc12_code": str,
            "huc10_code": str, "huc8_code": str, "mukey": str}
S2_CODES = {"county_fips": str, "huc12_code": str, "huc8_code": str}

## Step 1: Load S1 (base) + derive `year`

S1 is the base grain — one row per `(station, timestamp)`. `year` is extracted
from `ActivityStartDateTime` to key the annual S2 context.

In [2]:
df = pd.read_csv(
    f"{SECONDARY}/wq-geo-soil-daily.csv",
    parse_dates=["ActivityStartDateTime"],
    dtype=S1_CODES,
    low_memory=False,
)
print(f"S1 wq-geo-soil-daily: {df.shape}")
assert not df.duplicated(subset=GRAIN).any(), "S1 grain violated: duplicate (station, timestamp) rows"

df["year"] = df["ActivityStartDateTime"].dt.year
print(f"Measurement years: {df['year'].min()}-{df['year'].max()}")
n_rows = len(df)

S1 wq-geo-soil-daily: (48251, 102)
Measurement years: 2015-2025


## Step 2: Load S2, drop columns S1 already carries

The membership keys S2 shares with S1 (`county_fips`, `huc12_code`, `huc8_code`)
are dropped from the S2 side — except `MonitoringLocationIdentifier` and `year`,
which are the join keys — so the merge contributes only the new annual context.

In [3]:
s2 = pd.read_csv(f"{SECONDARY}/station-year-context.csv", dtype=S2_CODES)
assert not s2.duplicated(subset=[KEY, "year"]).any(), "S2 is not 1 row per station+year"

overlap = [c for c in s2.columns if c in df.columns and c not in (KEY, "year")]
s2 = s2.drop(columns=overlap)
new_cols = [c for c in s2.columns if c not in (KEY, "year")]
print(f"S2 station-year-context: shape after drop {s2.shape}")
print(f"Dropped {len(overlap)} overlapping columns kept from S1: {overlap}")
print(f"Adding {len(new_cols)} new annual-context columns")

S2 station-year-context: shape after drop (18326, 214)
Dropped 3 overlapping columns kept from S1: ['county_fips', 'huc12_code', 'huc8_code']
Adding 212 new annual-context columns


## Step 3: Merge S2 onto S1 (left join on station + year)

In [4]:
df = df.merge(s2, on=[KEY, "year"], how="left")

print(f"Merged shape: {df.shape}")
print(f"Annual context matched: {df['pct_corn'].notna().mean():.1%} of measurement rows")
assert len(df) == n_rows, "S2 join fanned out measurement rows"

Merged shape: (48251, 315)
Annual context matched: 100.0% of measurement rows


## Step 4: Final checks and save

In [5]:
print(f"Final shape: {df.shape}")
assert not df.duplicated(subset=GRAIN).any(), "Output grain violated: duplicate (station, timestamp) rows"
assert len(df) == n_rows, "Row count changed vs. S1 base"

print("\nCoverage (share of measurement rows):")
print(f"  Soil attributes:       {df['hydrologic_group'].notna().mean():.1%}")
print(f"  PRISM climate:         {df['prism_tmax_c'].notna().mean():.1%}")
print(f"  Streamflow discharge:  {df['streamflow_discharge_cfs'].notna().mean():.1%}")
print(f"  Land use (P5):         {df['pct_corn'].notna().mean():.1%}")
print(f"  NPDES (P6):            {df['npdes__n_facilities'].notna().mean():.1%}")
print(f"  Crop/livestock (P3):   {df['cropyield__corn_grain__yield__survey'].notna().mean():.1%}")
print(f"  N&P nutrients (P3b):   {df['npfert__n__total_kg'].notna().mean():.1%}")
print(f"  State spending (P4):   {df['spend__feed__usd'].notna().mean():.1%}")
df.head(3)

Final shape: (48251, 315)

Coverage (share of measurement rows):
  Soil attributes:       63.0%
  PRISM climate:         96.6%
  Streamflow discharge:  50.0%
  Land use (P5):         100.0%
  NPDES (P6):            100.0%
  Crop/livestock (P3):   94.5%
  N&P nutrients (P3b):   100.0%
  State spending (P4):   95.9%


,MonitoringLocationIdentifier,ActivityStartDateTime,"Temperature, water_value","Temperature, water_unit",Dissolved oxygen (DO)_value,Dissolved oxygen (DO)_unit,pH_value,pH_unit,Nitrate_value,Nitrate_unit,...,spend__chemical_totals__usd,spend__chemical_totals__usd_per_operation,spend__feed__pct_of_operations,spend__feed__pct_of_prod_expenses,spend__feed__usd,spend__feed__usd_per_operation,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_operations,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_prod_expenses,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd_per_operation
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2017-07-18 15:00:00,19.820,deg C,8.75,mg/L,8.330,std units,NaN,NaN,...,1.140000e+09,13240.0,40.9,16.7,4.400000e+09,51103.0,60.2,6.9,1.810000e+09,21022.0
1,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2023-07-19 11:00:00,16.372,deg C,12.24,mg/L,8.215,std units,NaN,NaN,...,1.950000e+09,22596.0,31.6,21.2,8.040000e+09,93163.0,60.1,8.7,3.280000e+09,38007.0
2,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2017-07-17 17:00:00,19.920,deg C,7.62,mg/L,8.195,std units,NaN,NaN,...,1.140000e+09,13240.0,40.9,16.7,4.400000e+09,51103.0,60.2,6.9,1.810000e+09,21022.0


In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 48,251 rows x 315 cols -> ../../data/03c_merge_tertiary/epa-full.csv
